In [29]:
import pandas as pd

In [30]:
df1 = pd.read_csv('main_result.csv')
df1 = df1[df1['n_self_alignment'] == 5]

df2 = pd.read_csv('feedback_result.csv')
df = pd.concat([df1, df2], axis=0)
df = df[df['gpt_model'] == 'gpt-4o']
# df = df[df['pe'].isin(['cot'])]
df

,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,Evaluation/reach_imp_perc,Evaluation/path_length,Evaluation/fn_imp_perc,Evaluation/fp_imp_perc,Evaluation/tn_imp_perc,Evaluation/tp_imp_perc,Evaluation/solvability,Evaluation/playability,score,Evaluation/naive_playability
168,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.666667,27.846155,2.500000,0.4,0.000000,0.100000,0.200000,0.433333,0.033333,NaN
169,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.233333,0.000000,3.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
170,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.433333,26.750000,2.733334,0.0,0.200000,0.066667,0.100000,0.266667,0.088889,NaN
171,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.366667,26.285715,2.833333,0.0,0.133333,0.033333,0.066667,0.233333,0.055556,NaN
172,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.700000,26.700001,1.800000,0.0,0.866667,0.333333,0.433333,0.666667,0.400000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
805,l06541ef,finished,6,cot,gpt-4o,2,feedback,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000
806,l06541ef,finished,6,cot,gpt-4o,2,feedback,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000
807,l06541ef,finished,6,cot,gpt-4o,2,feedback,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000
808,l06541ef,finished,6,cot,gpt-4o,2,feedback,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000


In [31]:
# replace feedback_type no -> No, generic -> Generic, default -> Specific
df['feedback_type'] = df['feedback_type'].replace({'no': 'no', 'generic': 'generic', 'default': 'specific'})

In [32]:
# if evaluator == 'llm' and target_character in [5, 6], and seed > 5 then drop
df = df[~((df['feedback_type'].isin(['no', 'generic'])) & (df['target_character'].isin([5, 6])) & (df['seed'] > 4))]

In [33]:
df.groupby(['pe', 'target_character', 'feedback_type']).agg({'Evaluation/llm_iteration': ['count']})

Evaluation/llm_iteration
                                                      count
pe  target_character feedback_type                         
cot 1                generic                             24
                     no                                  30
                     specific                            30
    2                generic                             24
                     no                                  30
                     specific                            30
    5                generic                             30
                     no                                  30
                     specific                            30
    6                generic                             30
                     no                                  30
                     specific                            30
got 1                generic                             30
                     no                                  18
                     specific                            30
    2                generic                             30
                     no                                  24
                     specific                            30
    5                generic                             30
                     no                                  30
                     specific                            30
    6                generic                             30
                     no                                  30
                     specific                            30
tot 1                generic                             30
                     no                                  18
                     specific                            30
    2                generic                             30
                     no                                  18
                     specific                            30
    5                generic                             30
                     no                                  30
                     specific                            30
    6                generic                             30
                     no                                  30
                     specific                            30

In [34]:
# min-max normalization with 'score' column for target_character-wise
def min_max_normalize(df):
    df = df.copy()
    def normalize_group(sub_df):
        min_val = sub_df['score'].min()
        max_val = sub_df['score'].max()
        if max_val > min_val:
            sub_df['score'] = (sub_df['score'] - min_val) / (max_val - min_val)
        else:
            sub_df['score'] = 0.0  # 모든 값이 동일한 경우
        return sub_df
    return df.groupby('target_character').apply(normalize_group).reset_index(drop=True)

normalized = min_max_normalize(df)
normalized

/var/folders/x_/2lt9k5kn52q43m1_z0kp7tfm0000gn/T/ipykernel_26126/3838729616.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('target_character').apply(normalize_group).reset_index(drop=True)


,run_id,final_state,target_character,pe,gpt_model,branch_factor,exp_name,evaluator,total_iterations,n_self_alignment,...,Evaluation/reach_imp_perc,Evaluation/path_length,Evaluation/fn_imp_perc,Evaluation/fp_imp_perc,Evaluation/tn_imp_perc,Evaluation/tp_imp_perc,Evaluation/solvability,Evaluation/playability,score,Evaluation/naive_playability
0,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.666667,27.846155,2.500000,0.4,0.000000,0.100000,0.200000,0.433333,0.042857,NaN
1,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.233333,0.000000,3.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
2,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.433333,26.750000,2.733334,0.0,0.200000,0.066667,0.100000,0.266667,0.114286,NaN
3,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.366667,26.285715,2.833333,0.0,0.133333,0.033333,0.066667,0.233333,0.071429,NaN
4,3kk20v1f,finished,1,cot,gpt-4o,2,sa,hr,6,5,...,0.700000,26.700001,1.800000,0.0,0.866667,0.333333,0.433333,0.666667,0.514286,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1021,ye6cp87h,finished,6,cot,gpt-4o,2,feedback,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.766667,0.733333,0.733333,0.733333
1022,ye6cp87h,finished,6,cot,gpt-4o,2,feedback,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.966667,0.766667,0.766667,0.966667
1023,ye6cp87h,finished,6,cot,gpt-4o,2,feedback,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,0.933333,0.900000,0.900000,0.900000
1024,ye6cp87h,finished,6,cot,gpt-4o,2,feedback,hr,6,5,...,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.100000,0.100000,1.000000


In [35]:
# normalized = pd.concat([normalized, mean_df], ignore_index=True)
normalized = pd.concat([normalized], ignore_index=True)
normalized['pe'].unique()

array(['cot', 'tot', 'got'], dtype=object)

In [36]:
avg_df = normalized.groupby(['gpt_model', 'pe', 'Evaluation/llm_iteration', 'feedback_type', 'seed']).agg({'score': 'mean'}).reset_index()
avg_df

,gpt_model,pe,Evaluation/llm_iteration,feedback_type,seed,score
0,gpt-4o,cot,1,generic,0,0.119048
1,gpt-4o,cot,1,generic,1,0.319923
2,gpt-4o,cot,1,generic,2,0.637890
3,gpt-4o,cot,1,generic,3,0.404557
4,gpt-4o,cot,1,generic,4,0.368062
...,...,...,...,...,...,...
343,gpt-4o,tot,6,specific,5,0.033005
344,gpt-4o,tot,6,specific,6,0.564286
345,gpt-4o,tot,6,specific,7,0.337165
346,gpt-4o,tot,6,specific,8,1.000000


In [37]:
df = pd.concat([avg_df], ignore_index=True)

In [38]:
iterwise_df = avg_df.groupby(
    ['pe', 'feedback_type', 'Evaluation/llm_iteration']
).agg({'score': ['mean', 'std']}).reset_index()

iterwise_df.columns = ['pe', 'feedback_type', 'Evaluation/llm_iteration', 'score', 'std']

iterwise_df = iterwise_df.pivot(
    index=['pe', 'feedback_type'],
    columns='Evaluation/llm_iteration',
    values=['score', 'std']
).reset_index()
iterwise_df.columns = iterwise_df.columns.remove_unused_levels()
iterwise_df.columns.names = [None, None]  # 레벨 이름 제거
iterwise_df.columns = [
    f"{col[0]}_{col[1]}" if col[1] != '' else col[0]
    for col in iterwise_df.columns
]
iterwise_df

,pe,feedback_type,score_1,score_2,score_3,score_4,score_5,score_6,std_1,std_2,std_3,std_4,std_5,std_6
0,cot,generic,0.369896,0.457813,0.536798,0.312813,0.341007,0.481530,0.186043,0.170980,0.165569,0.171962,0.138113,0.127010
1,cot,no,0.377406,0.401831,0.404138,0.345624,0.467890,0.360739,0.086600,0.096779,0.175223,0.084822,0.143985,0.119582
2,cot,specific,0.161999,0.198720,0.364499,0.203591,0.160734,0.285197,0.206509,0.227230,0.268027,0.269473,0.122395,0.308197
3,got,generic,0.430608,0.372258,0.396108,0.399762,0.409442,0.428292,0.144495,0.089053,0.093602,0.147532,0.124631,0.064311
4,got,no,0.387069,0.301308,0.314434,0.294037,0.345832,0.280862,0.051868,0.126624,0.147622,0.095776,0.116333,0.090505
5,got,specific,0.009214,0.330530,0.302493,0.186842,0.416916,0.424448,0.021895,0.313199,0.328878,0.209378,0.362209,0.329937
6,tot,generic,0.466913,0.538924,0.535394,0.401814,0.372085,0.405246,0.187430,0.176727,0.123027,0.084444,0.091840,0.094123
7,tot,no,0.374652,0.195153,0.354362,0.277482,0.261489,0.387447,0.128963,0.114673,0.142896,0.091113,0.170767,0.276472
8,tot,specific,0.160119,0.187069,0.190858,0.340739,0.339286,0.360235,0.255320,0.221009,0.176446,0.341267,0.366753,0.312893


In [39]:
result_df = iterwise_df.sort_values(by=['pe', 'feedback_type'], key=lambda x: x.map({'cot': 0, 'tot': 1, 'got': 2, 'mean': 3, 'no': 0, 'generic': 1, 'specific': 2})).reset_index(drop=True)
result_df

,pe,feedback_type,score_1,score_2,score_3,score_4,score_5,score_6,std_1,std_2,std_3,std_4,std_5,std_6
0,cot,no,0.377406,0.401831,0.404138,0.345624,0.467890,0.360739,0.086600,0.096779,0.175223,0.084822,0.143985,0.119582
1,cot,generic,0.369896,0.457813,0.536798,0.312813,0.341007,0.481530,0.186043,0.170980,0.165569,0.171962,0.138113,0.127010
2,cot,specific,0.161999,0.198720,0.364499,0.203591,0.160734,0.285197,0.206509,0.227230,0.268027,0.269473,0.122395,0.308197
3,tot,no,0.374652,0.195153,0.354362,0.277482,0.261489,0.387447,0.128963,0.114673,0.142896,0.091113,0.170767,0.276472
4,tot,generic,0.466913,0.538924,0.535394,0.401814,0.372085,0.405246,0.187430,0.176727,0.123027,0.084444,0.091840,0.094123
5,tot,specific,0.160119,0.187069,0.190858,0.340739,0.339286,0.360235,0.255320,0.221009,0.176446,0.341267,0.366753,0.312893
6,got,no,0.387069,0.301308,0.314434,0.294037,0.345832,0.280862,0.051868,0.126624,0.147622,0.095776,0.116333,0.090505
7,got,generic,0.430608,0.372258,0.396108,0.399762,0.409442,0.428292,0.144495,0.089053,0.093602,0.147532,0.124631,0.064311
8,got,specific,0.009214,0.330530,0.302493,0.186842,0.416916,0.424448,0.021895,0.313199,0.328878,0.209378,0.362209,0.329937


In [43]:
import pandas as pd
import numpy as np

def iterwise_df_to_latex(df: pd.DataFrame) -> str:
    """
    iterwise_df를 LaTeX 테이블로 변환.
    - score/std 짝을 mean ± std 로 표시
    - 각 (pe, feedback_type) row에서 최고 score iteration은 mean 부분만 bold
    - 표준편차는 \scriptsize 로 더 작게 표시
    - PE가 바뀔 때마다 \midrule 삽입
    """
    latex_rows = []
    prev_pe = None

    # iteration 번호 추출
    score_cols = sorted([c for c in df.columns if c.startswith("score_")],
                        key=lambda x: int(x.split("_")[1]))
    std_cols = sorted([c for c in df.columns if c.startswith("std_")],
                      key=lambda x: int(x.split("_")[1]))
    iterations = [c.split("_")[1] for c in score_cols]
    
    # $y_{i}$
    iterations = [f"$y_{{{i}}}$" for i in iterations]

    for _, row in df.iterrows():
        row_pe = str(row["pe"]).replace('cot', 'CoT').replace('tot', 'ToT').replace('got', 'GoT').replace('mean', 'Mean')  
        feedback_type = str(row["feedback_type"]).capitalize()
        row_entries = [row_pe, feedback_type]
        # capitalize first letter of feedback_type

        # 최고 score 컬럼 찾기
        best_col = row[score_cols].astype(float).idxmax()

        for s, st in zip(score_cols, std_cols):
            mean_val = row[s]
            std_val = row[st]

            # 기본 표현: mean ± std (std는 작은 글씨)
            # base = f"{mean_val:.3f}$\\pm${{\\scriptsize {std_val:.3f}}}"
            base = f"{mean_val:.3f}"

            # 최고 mean만 bold 처리 (std는 그대로)
            if s == best_col:
                base = f"\\textbf{{{mean_val:.3f}}}"

            row_entries.append(base)

        # pe가 바뀔 때 midrule 삽입
        if prev_pe is not None and row_pe != prev_pe:
            latex_rows.append("\\midrule")
        prev_pe = row_pe

        latex_rows.append(" & ".join(row_entries) + " \\\\")

    # --- colspec (동적으로) ---
    col_spec = "p{1.5cm}p{1.5cm}" + " c" * len(iterations)

    # --- header: iteration 번호만 표시 ---
    header = ["PE", "Feedback"] + iterations

    # --- Table build ---
    table = ""
    table += "\\begin{table}\n"
    table += "\\centering\n"
    table += "\\caption{Iteration-wise performance by PE and feedback type. Highest iteration per row is bold.}\n"
    table += "\\label{tab:iterwise_performance}\n"
    table += f"\\begin{{tabular}}{{{col_spec}}}\n"
    table += "\\toprule\n"
    table += " & ".join(header) + " \\\\\n"
    table += "\\midrule\n"
    table += "\n".join(latex_rows) + "\n"
    table += "\\bottomrule\n"
    table += "\\end{tabular}\n"
    table += "\\end{table}\n"
    return table


In [44]:
latex_str = iterwise_df_to_latex(result_df)
print(latex_str)

\begin{table}
\centering
\caption{Iteration-wise performance by PE and feedback type. Highest iteration per row is bold.}
\label{tab:iterwise_performance}
\begin{tabular}{p{1.5cm}p{1.5cm} c c c c c c}
\toprule
PE & Feedback & $y_{1}$ & $y_{2}$ & $y_{3}$ & $y_{4}$ & $y_{5}$ & $y_{6}$ \\
\midrule
CoT & No & 0.377 & 0.402 & 0.404 & 0.346 & \textbf{0.468} & 0.361 \\
CoT & Generic & 0.370 & 0.458 & \textbf{0.537} & 0.313 & 0.341 & 0.482 \\
CoT & Specific & 0.162 & 0.199 & \textbf{0.364} & 0.204 & 0.161 & 0.285 \\
\midrule
ToT & No & 0.375 & 0.195 & 0.354 & 0.277 & 0.261 & \textbf{0.387} \\
ToT & Generic & 0.467 & \textbf{0.539} & 0.535 & 0.402 & 0.372 & 0.405 \\
ToT & Specific & 0.160 & 0.187 & 0.191 & 0.341 & 0.339 & \textbf{0.360} \\
\midrule
GoT & No & \textbf{0.387} & 0.301 & 0.314 & 0.294 & 0.346 & 0.281 \\
GoT & Generic & \textbf{0.431} & 0.372 & 0.396 & 0.400 & 0.409 & 0.428 \\
GoT & Specific & 0.009 & 0.331 & 0.302 & 0.187 & 0.417 & \textbf{0.424} \\
\bottomrule
\end{tabular}
\end{t